# D3 · Change detection: the 2021 Melamchi flood

> **Time:** about 3 hours  
> **You need:** Parts A, B and C, and Lessons D1 and D2: bands with names and reflectance (D1), true colour with the same stretch for two dates (D1), a normalised difference with safe division, NDVI and MNDWI, masks and km² from 20 m cells (D2). From Part C: `reproject_match` (C4), slope and moving windows with `scipy.ndimage` (C6), and `features.shapes` (C7). From Part B: `dissolve` (B6) and map elements (B11).  
> **By the end you can:**
> 1. map change between two dates by **image differencing**, choose a **threshold** from the histogram, and report how much the area depends on it,
> 2. recognise **false change** (shadows, specks, two satellites) and remove it with `scipy.ndimage` and a **terrain rule** that you state and test,
> 3. turn the change mask into polygons, measure their area and length along the valley, save them to a GeoPackage, and make an honest final map.

In **June 2021** a debris flow and flood swept down the Melamchi river in **Sindhupalchok district**. It buried parts of Melamchi Bazar and the valley floor under sediment. You looked at the valley before and after in D1, and measured vegetation with NDVI in D2. This lesson asks the geohazard question: **where did the ground change, and how much of it?**

There are no field measurements in this lesson. Every area you compute is **your own image-based estimate**, and the lesson spends as much time on how wrong it could be as on the number itself. That is what a careful hazard analyst does.

The two images were downloaded once by `../data/get_sentinel2_melamchi.py` from a STAC catalogue (D1). Nothing here needs the internet.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.* The DEM keeps its own credit line where it is used.

**Counting lines:** lines that only set up or show a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**About the numbers:** the DEM is resampled onto the image grid, and resampling can differ very slightly between library versions. Numbers that use the DEM may differ from the text in the last digit.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 36 of 39 · [Course home](../README.md)  
← Previous: [D2 · Spectral indices: vegetation, water and bare ground in the Melamchi valley](D02_Spectral_indices.ipynb) · Next: [D4 · Land-cover classification: forest, fields, sediment and water](D04_Land_cover_classification.ipynb) →

## 1. Two dates on one grid

**Change detection** compares the same place at two dates and maps what differs. The simplest way compares the two images **cell by cell**.

That is only fair if both images sit on exactly the **same grid**: the same CRS, the same transform and the same shape (C1). Otherwise cell [500, 300] of one image is a different patch of ground in the other, and every "change" is partly a shift.

`get_sentinel2_melamchi.py` put both images on one 20 m grid in UTM 45N. Check it anyway: a check costs three lines, a wrong comparison costs the whole analysis.

The first cell gathers the tools of D1 and D2 into four short functions, so this lesson can stand on its own:

- `open_s2(path)`: opens an image with named bands, nodata as `NaN`, and multiplies by 0.0001 (D1, D2).
- `get(image, name)`: one band as a NumPy array, for the grid maths of this lesson.
- `composite(image, band_names, tops)`: three bands as a picture, divided by `tops` and clipped to 0–1 (D1).
- `normalized_difference(a, b)`: (a − b) / (a + b), with `NaN` where the sum is 0 (D2).

Run it first: every later cell needs it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import rioxarray as rxr
from rasterio import features
from rasterio.enums import Resampling
from matplotlib.colors import ListedColormap
from scipy import ndimage
from shapely.geometry import shape, LineString

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)
BEFORE = DATA / "s2_melamchi_before.tif"
AFTER = DATA / "s2_melamchi_after.tif"
S2_CREDIT = "Contains modified Copernicus Sentinel data 2020–2021, processed by ESA."


def open_s2(path):
    """Open a course Sentinel-2 file as reflectance (float32, NaN for nodata), with named bands (D1, D2)."""
    image = rxr.open_rasterio(path, masked=True)
    image = image.assign_coords(band=list(image.attrs["long_name"])) * 0.0001   # offset 0 for these dates
    image.attrs = {}      # the file's old notes (scale factor) no longer describe these values (D2)
    return image


def get(image, name):
    """One band as a NumPy array."""
    return image.sel(band=name).values


def composite(image, band_names, tops=None):
    """Three bands as an RGB picture, each divided by its top value and clipped to 0-1 (D1)."""
    rgb = np.dstack([get(image, name) for name in band_names])
    if tops is None:
        tops = np.nanpercentile(rgb, 98, axis=(0, 1))
    return np.clip(np.nan_to_num(rgb / tops), 0, 1)


def normalized_difference(a, b):
    """(a - b) / (a + b) for each cell, with NaN where a + b is 0 (D2)."""
    total = a + b
    total = np.where(total == 0, np.nan, total)     # guard: no division by zero
    return (a - b) / total

In [ ]:
before = open_s2(BEFORE)
after = open_s2(AFTER)

with rasterio.open(BEFORE) as src_b, rasterio.open(AFTER) as src_a:
    print("same CRS:      ", src_b.crs == src_a.crs, src_b.crs)
    print("same transform:", src_b.transform == src_a.transform)
    print("same shape:    ", src_b.shape == src_a.shape, src_b.shape)
    transform = src_b.transform
    left, bottom, right, top = src_b.bounds

extent_km = [left / 1000, right / 1000, bottom / 1000, top / 1000]
CELL_KM2 = 20 * 20 / 1000000            # one 20 m cell is 0.0004 km² (D2)
print(before.band.values)

scenes = pd.read_csv(DATA / "s2_melamchi_scenes.csv")
print(scenes[["image", "date", "scene_id"]].to_string(index=False))

In [ ]:
TRUE_COLOUR = ["B04", "B03", "B02"]
tops = np.nanpercentile(np.dstack([get(before, b) for b in TRUE_COLOUR]), 98, axis=(0, 1))   # one stretch for both dates (D1)

fig, axes = plt.subplots(1, 2, figsize=(12, 9))   # fig: the whole figure; axes: its two map panels (A10)
for ax, image, title in zip(axes, [before, after], ["Before: 29 October 2020", "After: 11 November 2021"]):
    ax.imshow(composite(image, TRUE_COLOUR, tops), extent=extent_km)
    ax.set_title(title)
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.suptitle("Melamchi valley, Sentinel-2 true colour, same stretch for both dates")
fig.text(0.01, 0.02, S2_CREDIT, fontsize=9)
plt.show()

Both files answer `True` three times: the same CRS (EPSG:32645), the same transform and the same shape, 1,006 rows × 702 columns. Cell [500, 300] is the same 20 m square of ground in both images, so subtracting them is fair.

The scene names start with the satellite: **S2A** took the before image on 29 October 2020, **S2B** the after image on 11 November 2021. Keep that in mind; step 5 comes back to it.

The pictures use the before image's 98th percentiles for both dates, as in D1, so equal colours mean equal reflectances. In 2020 the river is a narrow grey thread. In 2021 a **wide white strip** of sediment follows the valley floor from the northern edge to the southern edge. Anything that bright is above the 98th percentile and shows as pure white.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** How many cells of the **after** image's red band (B04) are `NaN`? Use `get`, `np.isnan` and `.sum()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
int(np.isnan(get(after, "B04")).sum())
```

125 cells: the zero values in deep shadow that D1 found, now `NaN` because `masked=True` treated 0 as nodata. Every index that uses B04 will be `NaN` there too.
</details>

**Q2 · two lines:** Add a column `satellite` to `scenes` with the first three characters of `scene_id` (`.str[:3]`, B5), and print `image`, `date` and `satellite`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
scenes["satellite"] = scenes["scene_id"].str[:3]
print(scenes[["image", "date", "satellite"]])
```

`before` was taken by S2A and `after` by S2B. The two satellites carry identical instruments, but two instruments are never perfectly identical: a small source of false change.
</details>

**Q3 · three lines:** Is the after image darker overall? Print the mean near-infrared reflectance (B08) of each image with `np.nanmean`, and the ratio after / before, each rounded to 3 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
nir_mean_before = np.nanmean(get(before, "B08"))
nir_mean_after = np.nanmean(get(after, "B08"))
print(round(nir_mean_before, 3), round(nir_mean_after, 3), round(nir_mean_after / nir_mean_before, 3))
```

About 0.300 before and 0.247 after: the whole after image reflects about 18% less near-infrared light (ratio about 0.824). The flood strip is far too small to cause that. A lower sun, a different satellite and a different atmosphere all change the brightness of a whole scene.
</details>

## 2. Image differencing: after minus before

The simplest change detection is **image differencing**: compute the same index on both dates, then subtract, cell by cell:

**change = index after − index before**

With **NDVI** (B08 and B04, D2), green vegetation scores high and bare sediment, rock and water score low. Where sediment buried fields and forest, NDVI **falls**, so the difference is **negative**.

A second index, built from **other bands**, is a useful witness. **MNDWI** (green B03 and short-wave infrared B11, D2) is strongly negative for green vegetation, and higher for bare sediment, wet ground and water. Where NDVI fell **and** MNDWI rose, two independent measurements tell the same story.

Show differences with a **diverging** colour map centred on 0 (D2): `RdBu` with `vmin=-0.6, vmax=0.6` makes losses red, gains blue and "no change" white.

**Predict:** before you run the cells, where do you expect the NDVI difference to be most negative? And will the mean difference of the whole image be large or small?

In [ ]:
ndvi_before = normalized_difference(get(before, "B08"), get(before, "B04"))
ndvi_after = normalized_difference(get(after, "B08"), get(after, "B04"))
d_ndvi = ndvi_after - ndvi_before

mndwi_before = normalized_difference(get(before, "B03"), get(before, "B11"))
mndwi_after = normalized_difference(get(after, "B03"), get(after, "B11"))
d_mndwi = mndwi_after - mndwi_before

print("mean NDVI   before:", round(float(np.nanmean(ndvi_before)), 3), "| after:", round(float(np.nanmean(ndvi_after)), 3))
print("mean change NDVI:", round(float(np.nanmean(d_ndvi)), 3), "| MNDWI:", round(float(np.nanmean(d_mndwi)), 3))
print("cells with a value:", int(np.isfinite(d_ndvi).sum()), "of", d_ndvi.size)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 7.5))
for ax, grid, title in zip(axes, [d_ndvi, d_mndwi], ["Change in NDVI", "Change in MNDWI"]):
    im = ax.imshow(grid, extent=extent_km, cmap="RdBu", vmin=-0.6, vmax=0.6)
    ax.set_title(title)
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.colorbar(im, ax=axes, shrink=0.6, label="After minus before (red = fall, blue = rise)")
fig.suptitle("Image differencing: 11 November 2021 minus 29 October 2020")
fig.text(0.01, 0.02, S2_CREDIT, fontsize=9)
plt.show()

The mean NDVI barely moves, from about 0.765 to about 0.738, and the mean change is only about −0.026. Most of the valley did not change, and averages hide a narrow strip.

The maps do not hide it. A **dark red ribbon** of falling NDVI runs along the river, from the northern edge to the southern edge. The MNDWI map shows the **same ribbon in blue**: two indices from different bands agree. Away from the river, the NDVI map is speckled with pale red, mostly in the south-east and south-west: small, weak falls that steps 4 and 5 investigate.

All but 128 of the 706,212 cells have a value; the others are the shadow cells with a missing band.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** What is the largest fall in NDVI anywhere in the image? Use `np.nanmin` on `d_ndvi`, wrapped in `float()` and rounded to 2 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float(np.nanmin(d_ndvi)), 2)
```

About −1.31: a cell that went from dense vegetation to a **negative** NDVI, which means less near-infrared than red, as over water or very wet ground. A difference of two NDVI values can range from −2 to +2.
</details>

**Q2 · two lines:** What percentage of the cells with a value saw NDVI **fall** by more than 0.3, and what percentage saw it **rise** by more than 0.3? Divide by `np.isfinite(d_ndvi).sum()` (C5).

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
valid = np.isfinite(d_ndvi).sum()
print(round((d_ndvi < -0.3).sum() / valid * 100, 2), round((d_ndvi > 0.3).sum() / valid * 100, 2))
```

About 1.88% fell by more than 0.3, but only about 0.19% rose by that much, ten times fewer. Change is lopsided: something removed vegetation on a scale that nothing balanced.
</details>

**Q3 · three lines:** Find the cell with the largest NDVI fall with `np.nanargmin` and `np.unravel_index` (A8). Print its row and column, its NDVI before and after, and its MNDWI change, rounded to 2 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = np.unravel_index(np.nanargmin(d_ndvi), d_ndvi.shape)
print(row, col, round(ndvi_before[row, col], 2), round(ndvi_after[row, col], 2))
print(round(d_mndwi[row, col], 2))
```

Row 186, column 339, in the northern part of the strip: NDVI fell from about 0.84 to about −0.47, and MNDWI rose by about 0.37. Both indices say the vegetation is gone. `np.nanargmin` skips `NaN` and returns a position in the flattened grid; `np.unravel_index` turns it back into a row and a column.
</details>

## 3. Choosing a threshold, and how much it matters

A difference map is continuous. To say "changed" or "not changed", you need a **threshold**: here, every cell whose NDVI fell by more than a chosen amount.

No law of nature gives that number. You choose it, so choose it with your eyes open:

- Look at the **histogram** of the differences. Most cells sit in a tall peak around 0: unchanged ground plus noise. Real change forms a long **tail** of large falls.
- A **log scale** on the y-axis (`ax.set_yscale("log")`) shows bars of 10 and of 100,000 cells on one plot, so the tail becomes visible.
- Try two or three sensible thresholds and report how the area changes. This **sensitivity** is part of your result, not a detail to hide.

The example compares −0.2, −0.3 and −0.4.

In [ ]:
THRESHOLDS = [-0.2, -0.3, -0.4]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(d_ndvi[np.isfinite(d_ndvi)], bins=np.arange(-1.4, 0.81, 0.02), color="grey")
for t, colour in zip(THRESHOLDS, ["gold", "darkorange", "firebrick"]):
    ax.axvline(t, color=colour, linewidth=2, label=f"threshold {t}")
ax.set_yscale("log")
ax.set_xlabel("Change in NDVI (after minus before)")
ax.set_ylabel("Number of cells (log scale)")
ax.legend()
plt.show()

rows = []
for t in THRESHOLDS:
    cells = int((d_ndvi < t).sum())
    rows.append({"threshold": t, "cells": cells, "area_km2": cells * CELL_KM2})
print(pd.DataFrame(rows).round(2))

In [ ]:
level = (d_ndvi < -0.2).astype(float) + (d_ndvi < -0.3) + (d_ndvi < -0.4)   # 0, 1, 2 or 3 thresholds passed
level[level == 0] = np.nan

three_colours = ListedColormap(["gold", "darkorange", "darkred"])      # one colour per level

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(get(after, "B08"), extent=extent_km, cmap="gray", vmin=0, vmax=0.5, alpha=0.5)   # pale background
im = ax.imshow(level, extent=extent_km, cmap=three_colours, vmin=0.5, vmax=3.5, interpolation="nearest")
cbar = fig.colorbar(im, ax=ax, shrink=0.6, ticks=[1, 2, 3])
cbar.ax.set_yticklabels(["below -0.2", "below -0.3", "below -0.4"])
ax.set_title("NDVI fall at three thresholds\n(background: NIR, B08, 11 Nov 2021)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
fig.text(0.01, 0.02, S2_CREDIT, fontsize=8)
plt.show()

The histogram peaks a little below 0 and falls steeply on both sides. On the left, a broad **shoulder** runs from about −0.4 to −1: a second population of cells, those that lost their vegetation. A threshold belongs where the shoulder separates from the peak, and −0.2 to −0.4 brackets that place.

The table is the honest headline of this step. At −0.2 about **12.33 km²** count as change, at −0.3 about **5.32 km²**, at −0.4 about **2.89 km²**. The "same" analysis gives areas that differ by a factor of four, only through the threshold.

The map shows why. The strip along the river passes all three thresholds (dark red). The scattered specks on the hillsides are mostly gold or orange: they pass the loose thresholds only. A loose threshold mostly adds specks; a strict one starts to eat into the strip.

`(d_ndvi < -0.2).astype(float) + (d_ndvi < -0.3) + (d_ndvi < -0.4)` adds three True/False grids: `True` counts as 1 (C5), so each cell gets the number of thresholds it passes. `ListedColormap` (from `matplotlib.colors`) makes a colour map from a list of colours, one per level; `alpha=0.5` makes the grey background pale so that the colours stand out.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** What area, in km², has an NDVI fall below **−0.25**? Round to 2 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round((d_ndvi < -0.25).sum() * CELL_KM2, 2)
```

About 7.92 km², between the −0.2 and −0.3 answers, as expected.
</details>

**Q2 · two lines:** A common rule of thumb puts the threshold at the **mean minus three standard deviations** of the difference. Compute it with `np.nanmean` and `np.nanstd`, then print it and the area below it, both rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
auto_t = np.nanmean(d_ndvi) - 3 * np.nanstd(d_ndvi)
print(round(auto_t, 2), round((d_ndvi < auto_t).sum() * CELL_KM2, 2))
```

About −0.32 and 4.6 km²: close to the −0.3 of the example. A rule of thumb gives a number you can defend, but it is still a choice: two or four standard deviations could be defended in the same way.
</details>

**Q3 · three lines:** How fast does the area grow as the threshold loosens? Loop over `np.arange(-0.5, -0.09, 0.1)` and print each threshold (rounded to 1 decimal) with its area in km² (rounded to 1 decimal).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for t in np.arange(-0.5, -0.09, 0.1):
    area_km2 = (d_ndvi < t).sum() * CELL_KM2
    print(round(t, 1), round(area_km2, 1))
```

About 1.8, 2.9, 5.3, 12.3 and 32.5 km² from −0.5 to −0.1. From −0.5 to −0.3 the area grows slowly; beyond −0.3 it explodes, because the threshold walks into the noise peak around 0. When a small step of the threshold doubles the area, you are measuring noise.
</details>

## 4. Cleaning the mask with scipy.ndimage

The mask `d_ndvi < -0.3` holds the strip and thousands of **specks**. `scipy.ndimage` has tools for True/False grids, called **morphology**. Like C6's moving windows, they look at each cell's neighbours:

- `ndimage.binary_opening(mask)`: first **erodes** (every patch loses its edge cells), then **dilates** (every patch grows back by one cell). Specks vanish. But so does any part of a patch narrower than about 3 cells.
- `ndimage.label(mask, structure=EIGHT)`: gives every connected **group** of `True` cells its own number, 1, 2, 3 … and returns that grid and the number of groups. `EIGHT = np.ones((3, 3))` lets cells join through corners as well as edges, like C7's `connectivity=8`.
- `ndimage.binary_dilation(mask, structure=EIGHT)`: grows every patch by one cell. Here it decides only **which pieces belong together**: two pieces separated by a gap of one cell touch after growing. The area still counts only cells that really changed.

Then drop every group with fewer than **50** changed cells (0.02 km²). Fifty is a choice; Q3 tests another.

**Predict:** opening removes specks of one or two cells. What will it do to the parts of the strip that are only one or two cells wide?

In [ ]:
EIGHT = np.ones((3, 3), dtype=bool)          # neighbours through edges and corners

change = d_ndvi < -0.3
opened = ndimage.binary_opening(change)
print("change cells:", change.sum(), "| after opening:", opened.sum())

labels, n_labels = ndimage.label(change, structure=EIGHT)
label_ids, label_sizes = np.unique(labels[labels > 0], return_counts=True)
print(n_labels, "groups,", (label_sizes == 1).sum(), "of them single cells; largest:", label_sizes.max(), "cells")

In [ ]:
grown = ndimage.binary_dilation(change, structure=EIGHT)    # every patch grows by one cell
groups, n_groups = ndimage.label(grown, structure=EIGHT)
groups = np.where(change, groups, 0)                          # keep group numbers on changed cells only
ids, counts = np.unique(groups[groups > 0], return_counts=True)
print(n_groups, "groups after joining near neighbours; largest:", counts.max(), "cells")

big_ids = ids[counts >= 50]
cleaned = np.isin(groups, big_ids)
print(len(big_ids), "groups of 50 cells or more:", cleaned.sum(), "cells =", round(cleaned.sum() * CELL_KM2, 2), "km²")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 7), sharey=True)
for ax, mask, title in zip(axes, [change, opened, cleaned], ["NDVI fall below -0.3", "after binary_opening", "groups of 50+ cells"]):
    ax.imshow(get(after, "B08"), extent=extent_km, cmap="gray", vmin=0, vmax=0.5, alpha=0.5)
    ax.imshow(np.where(mask, 1.0, np.nan), extent=extent_km, cmap="autumn", interpolation="nearest")
    ax.set_xlim(355, 361)
    ax.set_ylim(3076, 3088)
    ax.set_title(title)
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.text(0.01, 0.01, S2_CREDIT + " Background: NIR (B08), 11 Nov 2021.", fontsize=8)
plt.show()

**Opening** halves the mask, from 13,294 to 6,519 cells. Look at the middle map: the specks are gone, but so are long pieces of the strip in the south, where it is only one or two cells wide. A tool that removes noise can remove the signal too.

**Labelling** the raw mask finds 2,674 groups; 1,671 of them are single cells. The largest group has 4,870 cells, but the strip itself is broken into several groups by small gaps.

**Joining near neighbours** first leaves 1,682 groups, and the largest now holds 7,727 cells. Only 10 groups have 50 cells or more. Together they hold 8,550 cells, about **3.42 km²**. The right map keeps the strip whole, including its thin parts, and drops the specks.

`groups[groups > 0]` keeps only the numbered cells; `np.unique(..., return_counts=True)` counts the cells of each group (C6); `np.isin(groups, big_ids)` is `True` where a cell's group number is in the list, like `.isin` in B5.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** How large is the biggest group, in km²? Use `counts.max()` and `CELL_KM2`, rounded to 2 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(counts.max() * CELL_KM2, 2)
```

About 3.09 km²: one connected strip holds most of the change that survives cleaning.
</details>

**Q2 · two lines:** What share of the `cleaned` cells would opening have kept? Count the cells that are `True` in both `cleaned` and `opened`, and divide by `cleaned.sum()`. Print the percentage, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
both = (cleaned & opened).sum()
print(round(both / cleaned.sum() * 100, 1))
```

About 73.8%. Opening would have thrown away a quarter of the cells that clearly belong to large groups, mostly along the narrow parts of the strip.
</details>

**Q3 · three lines:** Try a stricter minimum: keep only groups of **250** changed cells or more. Print how many groups pass and their area in km², rounded to 2 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
big_250 = ids[counts >= 250]
cleaned_250 = np.isin(groups, big_250)
print(len(big_250), round(cleaned_250.sum() * CELL_KM2, 2))
```

Only 1 group, the main strip, with about 3.09 km²: about a tenth less than the 3.42 km² with 50 cells. The minimum size matters much less than the threshold did, which is good to know and good to report.
</details>

## 5. False change: shadows and slopes

Not every fall in NDVI is the flood. A difference that is not a change on the ground is called **false change**. Common sources here:

- **Illumination.** On 11 November the Sun stands lower in the sky than on 29 October, so shadows are longer. In deep shadow, reflectance is tiny, and small errors become large errors in NDVI.
- **Two satellites, two orbits.** The images come from S2A and S2B, and from different **relative orbits** (the orbit number is in the scene name). The view angle and the instrument differ slightly.
- **Season and year.** The after image is a year and 13 days later. Crops are harvested, grass dries, leaves change.

Two of these can be tested with the data. Shadows: count cells whose NIR reflectance is below 0.1 on each date. Terrain: put the DEM on the image grid with `reproject_match` (C4), compute slope with `np.gradient` and 20 m cells (C6), and compute the **height above the valley floor**: each cell's elevation minus the lowest elevation within about 500 m (`ndimage.minimum_filter(size=25)`, a C6 moving window).

In [ ]:
print("relative orbits:", scenes["scene_id"].str.split("_").str[3].tolist())

nir_before, nir_after = get(before, "B08"), get(after, "B08")
dark_before = nir_before < 0.1               # deep shadow: very little NIR comes back
dark_after = nir_after < 0.1
newly_dark = dark_after & ~dark_before       # ~ means "not" (C5)
print("dark cells: before", dark_before.sum(), "| after", dark_after.sum(), "| newly dark", newly_dark.sum())
print("median NIR of the after image:", round(float(np.nanmedian(nir_after)), 3))

lit = ~dark_before & ~dark_after
print("spread (std) of the NDVI change: newly dark", round(float(np.nanstd(d_ndvi[newly_dark])), 3),
      "| lit on both dates", round(float(np.nanstd(d_ndvi[lit])), 3))

In [ ]:
dem = rxr.open_rasterio(DATA / "dem_kathmandu_melamchi.tif", masked=True).squeeze("band", drop=True)
dem20 = dem.rio.reproject_match(before, resampling=Resampling.bilinear)   # onto the image grid (C4)
elev_m = dem20.values
print(dem.shape, "->", elev_m.shape, "| NaN cells:", int(np.isnan(elev_m).sum()))

dz_drow, dz_dcol = np.gradient(elev_m, 20, 20)                         # 20 m cells (C6)
slope_deg = np.degrees(np.arctan(np.sqrt(dz_drow ** 2 + dz_dcol ** 2)))
above_floor_m = elev_m - ndimage.minimum_filter(elev_m, size=25)       # height above the lowest cell within about 500 m

largest = groups == ids[counts.argmax()]           # the biggest group from step 4
specks = np.isin(groups, ids[counts < 10])          # groups of fewer than 10 cells
where = pd.DataFrame({
    "cells": [largest.sum(), specks.sum(), elev_m.size],
    "median_slope_deg": [float(np.median(slope_deg[m])) for m in [largest, specks, np.isfinite(slope_deg)]],
    "median_above_floor_m": [float(np.median(above_floor_m[m])) for m in [largest, specks, np.isfinite(above_floor_m)]],
}, index=["largest group", "groups under 10 cells", "whole image"])
print(where.round(1))

In [ ]:
specks_shown = ndimage.binary_dilation(specks)      # for display only: one-cell specks are too small to see

fig, ax = plt.subplots(figsize=(7, 9))
im = ax.imshow(above_floor_m, extent=extent_km, cmap="gray", vmin=0, vmax=200)
ax.imshow(np.where(specks_shown, 1.0, np.nan), extent=extent_km, cmap=ListedColormap(["red"]), interpolation="nearest")
ax.imshow(np.where(largest, 1.0, np.nan), extent=extent_km, cmap=ListedColormap(["cyan"]), interpolation="nearest")
fig.colorbar(im, ax=ax, shrink=0.6, extend="max", label="Height above the lowest cell within 500 m (m)")
ax.set_title("Largest group (cyan) and groups under 10 cells (red, enlarged)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
fig.text(0.01, 0.02, "Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018,\n"
         "provided under COPERNICUS by the European Union and ESA). " + S2_CREDIT, fontsize=7)
plt.show()

The relative orbits are **R076** (before) and **R119** (after): the two images were taken from different tracks in space.

**Shadows.** Deep-shadow cells grow from 3,132 before to 15,389 after, five times more. The median NIR of the after image is about 0.24, so 0.1 is truly dark. In the 13,043 newly dark cells, the NDVI change is almost twice as spread out (standard deviation about 0.174) as in cells lit on both dates (about 0.095). Shadows make NDVI noisy, and noise below a threshold looks like change.

**Terrain.** The DEM, 1,620 × 2,160 cells in degrees, became 1,006 × 702 cells on the image grid, with no gaps. The largest group lies on gentle ground (median slope about 10.3°), only about 14.2 m above the valley floor. The 3,197 cells of the groups under 10 cells lie on slopes of about 20.2°, a median of about 120 m above the valley floor, close to the whole image (26.8° and 132 m). The map shows it at a glance: one connected cyan strip in the black valley bottom, red specks scattered over the grey and white hillsides. The specks are drawn one cell larger than they are (`binary_dilation`, for display only), or you could not see them.

That is the pattern of this step: **the flood is a connected strip on the valley floor; weak false change is scattered on slopes.**

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** What percentage of the **newly dark** cells shows an NDVI fall below −0.3? Use a mask inside `d_ndvi[...]` and `.mean()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float((d_ndvi[newly_dark] < -0.3).mean() * 100), 1)
```

About 8.2%, more than four times the 1.88% of the whole image (step 2, Q2). New shadow alone makes a cell several times more likely to look "changed".
</details>

**Q2 · two lines:** How many of the speck cells (`specks`) lie on slopes steeper than 20°? Print the count and its percentage of all speck cells, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
steep_specks = (specks & (slope_deg > 20)).sum()
print(steep_specks, round(steep_specks / specks.sum() * 100, 1))
```

1,635 cells, about 51.1%: half of the speck cells lie on slopes steeper than 20°, against a median slope of only about 10° in the largest group.
</details>

**Q3 · three lines:** Prepare the rule of step 6. What percentage of the **whole image** lies less than 30 m above the valley floor, and what percentage of the cells of the **largest group**? Round both to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
low = above_floor_m < 30
print(round(low.mean() * 100, 1))
print(round(low[largest].mean() * 100, 1))
```

Only about 3.3% of the image, but about 84.7% of the largest group. A rule "less than 30 m above the valley floor" keeps most of the strip while excluding about 97% of the landscape.
</details>

## 6. A terrain rule, and an honest area

Step 5 turns into a **plausibility rule**: a group of changed cells counts as flood only if it lies on the valley floor. Write it down exactly:

> keep a group if the **median** height above the valley floor of its cells is below **30 m**.

Judge **whole groups**, not single cells. In a narrow gorge, some strip cells sit where the coarser DEM puts the valley wall; judged one by one they would be cut, and the strip would break up.

`ndimage.median(values, labels=groups, index=big_ids)` returns one median per group. It is **zonal statistics** (C7) with the groups as zones.

This rule is **a choice, not the truth**. The DEM shows the valley before the flood (its credit line names the years 2010 to 2018), 30 m is a round number, and a debris flow can run up a valley side. Say what you chose and why, and show what changes if you choose differently. The second example wraps steps 3 to 6 in one function, so the whole recipe runs for any threshold.

**Predict:** step 3 showed areas from 12.3 to 2.9 km² before cleaning. Will the rules of steps 4 to 6 make the three thresholds agree more, or less?

In [ ]:
median_above = np.array(ndimage.median(above_floor_m, labels=groups, index=big_ids))
group_table = pd.DataFrame({"group": big_ids, "cells": counts[counts >= 50], "median_above_floor_m": median_above.astype(float)})
print(group_table.round(1).to_string(index=False))

keep_ids = big_ids[median_above < 30]
flood = np.isin(groups, keep_ids)
print("kept", len(keep_ids), "of", len(big_ids), "groups:", flood.sum(), "cells =", round(flood.sum() * CELL_KM2, 2), "km²")

In [ ]:
def flood_mask(threshold, min_cells=50, max_above_m=30):
    """Steps 3 to 6: NDVI fall below threshold, grouped, big groups on the valley floor."""
    change = d_ndvi < threshold
    groups, n = ndimage.label(ndimage.binary_dilation(change, structure=EIGHT), structure=EIGHT)
    groups = np.where(change, groups, 0)
    ids, counts = np.unique(groups[groups > 0], return_counts=True)
    big = ids[counts >= min_cells]
    medians = np.array(ndimage.median(above_floor_m, labels=groups, index=big))
    return np.isin(groups, big[medians < max_above_m])


rows = []
for t in THRESHOLDS:
    rows.append({"threshold": t, "raw_km2": (d_ndvi < t).sum() * CELL_KM2, "final_km2": flood_mask(t).sum() * CELL_KM2})
area_table = pd.DataFrame(rows)
print(area_table.round(2))
print("the function repeats the steps:", bool((flood_mask(-0.3) == flood).all()))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(get(after, "B08"), extent=extent_km, cmap="gray", vmin=0, vmax=0.5, alpha=0.5)
for t, colour in zip(THRESHOLDS, ["gold", "darkorange", "darkred"]):
    ax.imshow(np.where(flood_mask(t), 1.0, np.nan), extent=extent_km, cmap=ListedColormap([colour]), interpolation="nearest")
ax.set_title("Final masks: gold -0.2, orange -0.3, dark red -0.4")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
fig.text(0.01, 0.02, S2_CREDIT, fontsize=8)
plt.show()

The group table makes the rule visible. Four groups lie a median of about 7.5 to 23.4 m above the valley floor: the main strip (group 17, 7,727 cells), its northern end (group 1) and two short pieces on the river that joins from the north-east. Five groups on the hillsides lie a median of about 114 to 181 m above it. One group, number 1430 with 75 cells, sits in between at about 42.5 m: it lies in a side valley in the south-west, far from the Melamchi river. The rule drops it. Is that right? The data cannot say; this is exactly the kind of case to list for a field check.

The rule keeps 4 of the 10 groups: 8,088 cells, about **3.24 km²**.

The function reproduces those cells exactly (`True`), and then gives the table you should report:

| threshold | raw area | after cleaning and terrain rule |
| --- | --- | --- |
| −0.2 | about 12.3 km² | about 4.9 km² |
| −0.3 | about 5.3 km² | about 3.2 km² |
| −0.4 | about 2.9 km² | about 2.2 km² |

Cleaning and the terrain rule shrink the spread from a factor of four to about two. A fair sentence for a report: *"Our image-based estimate of the area where vegetation was lost along the valley floor is about 3.2 km², and between about 2.2 and 4.9 km² for thresholds from −0.4 to −0.2."*

On the map, the three masks are drawn on top of each other: gold, then orange, then dark red. Most of the strip is dark red, so all three thresholds agree on it. The −0.2 mask adds a fringe along the strip and a longer stretch of the river that joins from the north-east. `ListedColormap([colour])` gives each mask one flat colour.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** Loosen the terrain rule: what is the area at −0.3 if groups may lie up to **60 m** above the valley floor? Round to 2 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(flood_mask(-0.3, max_above_m=60).sum() * CELL_KM2, 2)
```

About 3.27 km², hardly more than 3.24 km²: only the south-western group (about 42.5 m) joins. The other groups lie far below 30 m or far above 60 m, so at this threshold the result does not hang on the exact number. Review 2 shows where the rule does become fragile.
</details>

**Q2 · two lines:** Test the minimum group size of step 4: print the area at −0.3 for `min_cells` of 25 and of 100, rounded to 2 decimals, in a loop.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for m in [25, 100]:
    print(m, round(flood_mask(-0.3, min_cells=m).sum() * CELL_KM2, 2))
```

About 3.28 km² with 25 and 3.21 km² with 100. Like the terrain rule, the minimum size moves the answer by a few hundredths of a km². The threshold remains the choice that matters most.
</details>

**Q3 · three lines:** Cross-check with the second index. What percentage of the `flood` cells shows an MNDWI **rise** above 0.1? Compare it with the percentage of all cells with a value. Round both to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
in_flood = (d_mndwi[flood] > 0.1).mean() * 100
everywhere = (d_mndwi[np.isfinite(d_mndwi)] > 0.1).mean() * 100
print(round(in_flood, 1), round(everywhere, 1))
```

About 89.2% of the flood cells, against about 9.4% of the whole image. The flood mask was built from NDVI only, yet MNDWI, made from other bands, agrees in nine cells out of ten inside it: independent evidence that the change is real.
</details>

## 7. From mask to polygons: area, length and a GeoPackage

A mask lives on one grid. Polygons can be measured, shared and overlaid with any vector layer (Part B).

`features.shapes` turns the mask into polygons, as in C7: give it the mask as `uint8`, `mask=flood` to keep only the flood cells, and the image's `transform`. Then `dissolve()` (B6) merges all pieces into **one** feature, a MultiPolygon, to which you add columns that say what it is: dates, threshold, area and a note that it is an estimate.

**Length along the valley.** A winding strip has no single "length". One honest approximation is a **centreline**:

1. take the cell centres of the largest group (the main strip) as x and y in metres (C1's `xy`),
2. cut them into slices of 500 m of northing,
3. take the mean x and y of each slice with `groupby` (A9),
4. join those points, from north to south, into a `LineString` (B2) and measure it.

Finally, save the polygon to a **GeoPackage** in `../outputs` (B3).

In [ ]:
pieces_gen = features.shapes(flood.astype("uint8"), mask=flood, transform=transform)
pieces = gpd.GeoDataFrame(geometry=[shape(geom) for geom, value in pieces_gen], crs="EPSG:32645")
pieces["area_km2"] = pieces.area / 1000000
print(len(pieces), "polygons, total", round(pieces["area_km2"].sum(), 2), "km²")

flood_gdf = pieces.dissolve()                     # one row: all pieces as one MultiPolygon
flood_gdf["area_km2"] = flood_gdf.area / 1000000
flood_gdf["threshold"] = -0.3
flood_gdf["before"] = "2020-10-29"
flood_gdf["after"] = "2021-11-11"
flood_gdf["note"] = "NDVI fall, image-based estimate, not field-checked"
print(flood_gdf.drop(columns="geometry").round(2).to_string(index=False))

flood_gdf.to_file(OUT / "d03_melamchi_change.gpkg", layer="ndvi_fall_0p3", driver="GPKG")
print("saved:", (OUT / "d03_melamchi_change.gpkg").exists())

In [ ]:
main = flood & (groups == ids[counts.argmax()])           # the main strip only
rows_main, cols_main = np.nonzero(main)                     # row and column of every strip cell
xs, ys = rasterio.transform.xy(transform, rows_main, cols_main)   # cell centres in metres (C1)
cells = pd.DataFrame({"x": np.array(xs), "y": np.array(ys)})
cells["slice"] = cells["y"] // 500                          # 500 m slices of northing

centre = cells.groupby("slice")[["x", "y"]].mean().sort_values("y", ascending=False)
centreline = LineString(centre[["x", "y"]].to_numpy())
straight = LineString([centreline.coords[0], centreline.coords[-1]])
print(len(centre), "slices, from", round(centre["y"].max() / 1000, 1), "to", round(centre["y"].min() / 1000, 1), "km north")
print("along the centreline:", round(centreline.length / 1000, 1), "km | straight line:", round(straight.length / 1000, 1), "km")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
flood_gdf.plot(ax=ax, color="firebrick")
gpd.GeoSeries([centreline], crs="EPSG:32645").plot(ax=ax, color="black", linewidth=1)
ax.set_xlim(left, right)
ax.set_ylim(bottom, top)
ax.ticklabel_format(style="plain", useOffset=False)   # full numbers on the axes (C4)
ax.set_title("Change polygons (-0.3) and the centreline of the main strip")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

The mask becomes 412 polygons, because cells that touch only at a corner become separate pieces (`features.shapes` joins through edges only, C7), and the groups had gaps of one cell. Their areas add up to exactly the 3.24 km² of step 6: a good check. `dissolve()` makes them one feature with its attributes, and the GeoPackage layer `ndvi_fall_0p3` holds it for QGIS or any later lesson.

The centreline of the main strip runs through 40 slices, from about 3,095.0 km to about 3,075.9 km north: almost the full height of the image. It is about **21.8 km** long, against **19.5 km** in a straight line between its ends. A centreline of 500 m slices cuts across the river's smaller bends, so the river itself is longer; report the length together with the method.

The black line follows the red strip. Where the strip turns east and then south near the confluence, the line cuts the corner a little: 500 m slices cannot follow a strip that runs almost east–west. The red pieces in the north-east, off the line, are the two groups on the joining river.

With some library versions, `features.shapes` prints a `DeprecationWarning` about a "Memory" driver, as in C7; it is harmless.

### Practice

**Q1 · one line:** How many of the polygons are a **single cell**, 0.0004 km²? Count the rows of `pieces` with an area below 0.0005 km².

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(pieces["area_km2"] < 0.0005).sum()
```

223 of the 412 pieces are single cells: changed cells that touch the rest only at a corner or across a one-cell gap. Dissolving hides them in one MultiPolygon, but they are there.
</details>

**Q2 · two lines:** Read the layer back from the GeoPackage with `gpd.read_file(..., layer="ndvi_fall_0p3")` and print its CRS and `area_km2`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
back = gpd.read_file(OUT / "d03_melamchi_change.gpkg", layer="ndvi_fall_0p3")
print(back.crs, back["area_km2"].round(2).tolist())
```

EPSG:32645 and about 3.24 km²: the file kept the geometry, the CRS and the attributes.
</details>

**Q3 · three lines:** Estimate the **mean width** of the main strip: its area in m² (`main.sum()` cells × 400 m²) divided by the centreline length in m. Print the width in metres, rounded to 0 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
main_area_m2 = main.sum() * 400
width_m = main_area_m2 / centreline.length
print(round(width_m))
```

About 142 m, roughly 7 cells of 20 m. On a 20 m grid, a strip that narrow has many mixed edge cells, so the width is uncertain by a cell or two on each side. And because the centreline is shorter than the river, this width is, if anything, a little too large.
</details>

## 8. The final map

A map for a report puts everything of this lesson in one honest picture (B11):

- the **after image** in true colour, with its date in the title,
- the **change outline** in a colour that stands out against both green and white,
- a **scale bar** (`ScaleBar(dx=1, units="m")`, the map is in metres) and a **north arrow**,
- **Melamchi** marked at the town's position in OpenStreetMap, 85.5761° E, 27.8293° N, as in D1 (D5 works with OpenStreetMap), with its own credit line,
- a caption with the **credit line** and what the outline is: *your own image-based estimate, not field-checked*.

The outline comes from `flood_gdf.boundary` (B11). `bbox` gives text a background box so it stays readable on the image.

In [ ]:
from matplotlib_scalebar.scalebar import ScaleBar

melamchi = gpd.GeoSeries(gpd.points_from_xy([85.5761], [27.8293]), crs="EPSG:4326").to_crs(32645)   # from OpenStreetMap; see D5
extent_m = [left, right, bottom, top]

fig, ax = plt.subplots(figsize=(9, 12))
ax.imshow(composite(after, TRUE_COLOUR, tops), extent=extent_m)
flood_gdf.boundary.plot(ax=ax, color="magenta", linewidth=0.8)
melamchi.plot(ax=ax, color="yellow", marker="*", markersize=250, edgecolor="black", zorder=5)
ax.annotate("Melamchi", (melamchi.x.iloc[0], melamchi.y.iloc[0]), xytext=(-12, 10),
            textcoords="offset points", ha="right", fontsize=10, bbox={"facecolor": "white", "alpha": 0.8})
ax.add_artist(ScaleBar(dx=1, units="m", location="lower left", length_fraction=0.25))
ax.annotate("N", xy=(0.93, 0.96), xytext=(0.93, 0.88), xycoords="axes fraction", ha="center", va="center",
            fontsize=14, fontweight="bold", color="white",
            arrowprops={"facecolor": "white", "edgecolor": "black", "width": 4, "headwidth": 12})
ax.set_xlim(left, right)
ax.set_ylim(bottom, top)
ax.set_axis_off()
ax.set_title("Melamchi valley, 11 November 2021\nMagenta: NDVI fell by more than 0.3 since 29 October 2020 (on the valley floor)")
fig.text(0.13, 0.08, S2_CREDIT + "\nOutline: own image-based estimate (about 3.2 km²; 2.2 to 4.9 km² for thresholds -0.4 to -0.2), "
         "not field-checked.\nValley-floor rule uses the Copernicus DEM GLO-30 (credit in data/README.md). Melamchi: © OpenStreetMap contributors, ODbL.", fontsize=8)
fig.savefig(OUT / "d03_melamchi_change_map.png", dpi=300, bbox_inches="tight")
plt.show()

The magenta outline traces the white sediment strip from the northern edge of the image to the southern edge, plus a short stretch of the river that joins from the north-east. The scale bar shows how narrow the strip is compared with the valley, and the Melamchi star sits on the valley floor where the two rivers meet, right at the edge of the outline.

The caption carries three things a reader needs: the credit, what the outline is, and the range of areas. The 300 dpi PNG in `../outputs` is ready for a report.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** How far is the Melamchi mark from the change polygon? Use `.distance` (B2) between `melamchi.iloc[0]` and `flood_gdf.geometry.iloc[0]`, in metres, rounded to whole metres.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(melamchi.iloc[0].distance(flood_gdf.geometry.iloc[0]))
```

About 29 m: the town point lies just outside the outline. That does not mean the flood stopped 29 m from the town. A point marks a place, not its extent, and NDVI cannot fall where the ground was bare already, as on streets, roofs and the old gravel channel (section 10). The image itself shows the strip running past the town better than any single distance.
</details>

**Q2 · two lines:** How long is the **outline**? Print the length of `flood_gdf.boundary` in km (`.length`, B4), rounded to 1 decimal, and divide it by the centreline length.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
outline_km = flood_gdf.boundary.length.iloc[0] / 1000
print(round(outline_km, 1), round(outline_km / (centreline.length / 1000), 1))
```

About 168.3 km of outline for a strip whose centreline is about 21.8 km long: about 7.7 times as long. The outline follows the staircase edges of 20 m cells and runs around every hole and every small piece. A perimeter of a polygonised raster says more about the cells than about the land, so measure length along a centreline instead.
</details>

**Q3 · three lines:** Zoom in on the place where the two rivers meet on the **before** image (same `tops`), with the change outline and a title naming the date. `ax.axis([357500, 361000, 3077000, 3081500])` sets both axis limits in one line. What was there before?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(7, 8))
ax.imshow(composite(before, TRUE_COLOUR, tops), extent=extent_m)
flood_gdf.boundary.plot(ax=ax, color="magenta", linewidth=1)
ax.axis([357500, 361000, 3077000, 3081500])
ax.set_title("29 October 2020, with the change outline")
plt.show()
```

Inside the outline, the 2020 image shows green fields and trees beside a narrow grey channel. The channel itself is partly **outside** the outline: it was already bare in 2020, and NDVI cannot fall where nothing was green. Drawing the same outline on both dates is the clearest way to show a reader what changed.
</details>

## 9. When it breaks: two grids that do not match

You want to keep only change on low ground, and use the DEM directly, without putting it on the image grid first. Run the cell.

In [ ]:
low_ground = dem.values < 1500             # the DEM on its own grid, in degrees
flood_and_low = change & low_ground

Read the last line:

`ValueError: operands could not be broadcast together with shapes (1006,702) (1620,2160)`

- **operands** are the two things being combined: `change` and `low_ground`.
- **broadcast** is NumPy's way of combining arrays of different shapes (A8). It works only when the shapes fit, and (1006, 702) and (1620, 2160) do not.
- The shapes tell you what went wrong: the image grid has 1,006 × 702 cells of 20 m in UTM; the DEM has 1,620 × 2,160 cells of one arc-second in degrees.

The fix is `dem.rio.reproject_match(before, ...)`, as in step 5: it puts the DEM on the image grid.

A dangerous non-fix: cutting the DEM to 1,006 × 702 cells with slicing. The error would vanish, but cell [500, 300] would be a different place in each array, and every answer would be wrong without any warning. **Equal shapes are not the same grid.** Check the CRS and the transform too (step 1).

**Try it:** replace `dem.values` with `dem20.values` and run the cell again.

## 10. What this can and cannot tell you

**What it can tell you**

- Where green ground (fields, forest, bushes) became **bare or wet** between 29 October 2020 and 11 November 2021: a connected strip along the Melamchi valley floor, from the northern edge of the image to the southern edge.
- A first, image-based estimate of that area: about **3.2 km²**, between about **2.2 and 4.9 km²** depending on the threshold, with the cleaning and terrain rules stated.
- A map to guide **field checks** and to compare with other sources, such as reports, aerial photos or later images.

**What it cannot tell you**

- **Whether it is right.** There is no field validation here: nobody checked on the ground which cells really changed. Without that, you cannot give an accuracy, only a sensitivity.
- **When it happened.** Two single dates frame more than a year. The June 2021 flood is the obvious candidate for the strip, but any change in between (another flood, landslides, road works, gravel mining, a harvest) looks the same in a difference of two images.
- **What happened.** NDVI measures green vegetation. A fall in NDVI cannot separate **deposition** (sediment laid down) from **erosion** (ground scoured away) or from a river that moved. And ground that was already bare in 2020, like the old gravel channel, cannot lose NDVI: if new sediment buried it, NDVI does not see it. So the mask is **not** "the area buried by the flood". Review 1 looks at the old channel with MNDWI.
- **Exact edges.** On a 20 m grid, edge cells are mixed, and the width of a 150 m strip is uncertain by a cell or two on each side.
- **The terrain at the time.** The DEM shows the valley before the flood (its credit names the years 2010 to 2018), and the valley floor has changed since. The 30 m rule is a choice, not a law.

When you report this work, say all of it: the dates, the index, the threshold and its sensitivity, the rules, and the limits. A number with its uncertainty and its method is useful; a number on its own is not.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · What NDVI cannot see

Find cells on the valley floor (`above_floor_m < 30`) where MNDWI **rose** by more than 0.2 but which are **not** in `flood`. Print their number, their area in km² and their median NDVI **before** the flood. Then map them in blue with the flood mask in red, zoomed to x 355–361 km and y 3,076–3,088 km.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
missed = (above_floor_m < 30) & (d_mndwi > 0.2) & ~flood
print(missed.sum(), "cells,", round(missed.sum() * CELL_KM2, 2), "km²")
print("median NDVI before:", round(float(np.nanmedian(ndvi_before[missed])), 2))

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(get(after, "B08"), extent=extent_km, cmap="gray", vmin=0, vmax=0.5)
ax.imshow(np.where(flood, 1.0, np.nan), extent=extent_km, cmap="autumn", interpolation="nearest")
ax.imshow(np.where(missed, 1.0, np.nan), extent=extent_km, cmap="winter", interpolation="nearest")
ax.set_xlim(355, 361)
ax.set_ylim(3076, 3088)
ax.set_title("Red: NDVI fall (flood mask). Blue: MNDWI rise only")
plt.show()
```

1,092 cells, about 0.44 km², changed in MNDWI but not in NDVI. Their median NDVI before was only about 0.28, far below the image's mean of about 0.765 (step 2): they were **bare or sparsely vegetated already** in 2020. They lie inside and along the strip, and along the old channel south of the confluence. NDVI could not fall much there, so the NDVI mask misses ground that the flood may well have covered. A second index measures what the first cannot. `~flood` means "not in the flood mask" (C5).
</details>

### Review 2 · How much do the rules matter?

Use `flood_mask` to build a table of areas (km²) for the three thresholds of `THRESHOLDS` and three values of `max_above_m`: 15, 30 and 1,000 (1,000 m means "no terrain rule"). Use two loops, collect dictionaries in a list, and turn the list into a DataFrame. Show it with the thresholds as rows (`pivot`).

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
rows = []
for t in THRESHOLDS:
    for max_m in [15, 30, 1000]:
        area_km2 = flood_mask(t, max_above_m=max_m).sum() * CELL_KM2
        rows.append({"threshold": t, "max_above_m": max_m, "area_km2": area_km2})

table = pd.DataFrame(rows)
print(table.pivot(index="threshold", columns="max_above_m", values="area_km2").round(2))
```

Each row is one threshold; each column one terrain rule. Without a rule (1,000 m) the areas are about 7.84, 3.42 and 2.36 km²; with 30 m, 4.87, 3.24 and 2.24 km². The rule matters most at −0.2, where it removes about 3 km² of groups away from the valley floor.

The 15 m column is the surprise: about 0.06 km² at −0.2 and 0.34 km² at −0.4. The main strip lies a median of about 14.2 m above the valley floor (step 6), uncomfortably close to 15 m. At the other two thresholds the strip's cells are slightly different, its median creeps above 15 m, and the rule throws the **whole strip** away. A group rule is all or nothing, so set it well clear of the values it judges, as 30 m is. Testing your rule like this is how you find such traps. `pivot` (a pandas reshape) turns the long list into this grid.
</details>

### Review 3 · One GeoPackage, three layers

For each threshold in `THRESHOLDS`, polygonise `flood_mask(t)`, dissolve it, add columns `threshold` and `area_km2`, and save it to `d03_melamchi_change.gpkg` as a layer named like `ndvi_fall_0p2`. Then read each layer back and print its name and area.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
for t in THRESHOLDS:
    mask = flood_mask(t)
    polys = [shape(geom) for geom, value in features.shapes(mask.astype("uint8"), mask=mask, transform=transform)]
    layer_gdf = gpd.GeoDataFrame(geometry=polys, crs="EPSG:32645").dissolve()
    layer_gdf["threshold"] = t
    layer_gdf["area_km2"] = layer_gdf.area / 1000000
    name = "ndvi_fall_0p" + str(round(-t * 10))
    layer_gdf.to_file(OUT / "d03_melamchi_change.gpkg", layer=name, driver="GPKG")
    back = gpd.read_file(OUT / "d03_melamchi_change.gpkg", layer=name)
    print(name, round(back["area_km2"].iloc[0], 2))
```

Three layers, with about 4.87, 3.24 and 2.24 km². The `0p3` layer is overwritten with the same content as in step 7. `round(-t * 10)` turns −0.2 into 2 for the layer name; a dot in a layer name is legal but awkward in many tools, hence `0p2`. One file with one layer per threshold lets a reader see the sensitivity in QGIS.
</details>

### Review 4 · Where is the strip widest?

Using `cells` from step 7, cut the main strip into **1 km** slices of northing instead of 500 m. For each slice, the width in metres is the number of cells × 400 m² / 1,000 m. Print the three widest slices (northing of the slice start in km, and width in m), and draw the width of every slice as a horizontal bar chart against northing (A10).

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
cells["km_slice"] = cells["y"] // 1000
width_m = cells.groupby("km_slice").size() * 400 / 1000
width_m.index = width_m.index.astype(int)          # northing in whole km
print(width_m.sort_values(ascending=False).head(3).round(0))

fig, ax = plt.subplots(figsize=(6, 7))
ax.barh(width_m.index, width_m.values, color="firebrick")
ax.set_xlabel("Mean width of the main strip (m)")
ax.set_ylabel("Northing of the 1 km slice (km)")
ax.set_title("Width of the change strip along the valley (estimate)")
plt.show()
```

The widest slices start at 3,084 km (about 246 m), 3,079 km (about 220 m, by the confluence) and 3,088 km (about 200 m). The bars show where it narrows: in the gorge in the north, and between about 3,081 and 3,083 km. The slice at 3,075 km is only partly inside the image, so its width is far too small. Width per slice is a rough measure: where the strip runs diagonally, it crosses a slice at an angle and looks wider than it is.
</details>

## Recap

- **Image differencing**: put both dates on the same grid (check CRS, transform and shape), compute one index per date, subtract (after − before), and map the result with a diverging colour map centred on 0.
- A **threshold** is a choice. Read the histogram (log scale), try two or three values, and report the **sensitivity**: here 12.3, 5.3 and 2.9 km² before cleaning, for −0.2, −0.3 and −0.4.
- **False change** comes from shadows, different satellites and orbits, seasons and small shifts. In this valley it is scattered as specks on slopes, while the flood is one connected strip on the valley floor.
- `scipy.ndimage`: `binary_opening` removes specks but also thin parts; `label` numbers connected groups; `binary_dilation` joins pieces separated by small gaps; `median(..., labels=...)` summarises each group. A **terrain rule** (median height above the valley floor below 30 m) is stated, tested and reported as a choice.
- `features.shapes` + `dissolve` give polygons with attributes; a centreline gives a length along the valley; a GeoPackage stores the layers. The final map carries the date, the outline, a scale bar, north arrow, Melamchi (from OpenStreetMap), the credits and the words **image-based estimate, not field-checked**.

### Words you learned

| Word | Meaning |
| --- | --- |
| change detection | Finding where the ground changed between two dates |
| image differencing | Subtracting an index (or band) of one date from the same index of another date, cell by cell |
| threshold | The value that separates "changed" from "not changed" |
| sensitivity | How much a result changes when a choice, such as the threshold, changes |
| log scale | An axis on which each step multiplies the value, here by 10; shows small and huge counts together |
| false change | A difference between two images that is not a change on the ground |
| illumination | How the ground is lit by the Sun; it changes with date, time and slope |
| relative orbit | The numbered track along which a Sentinel-2 satellite passed when it took an image |
| morphology | Operations on the shape of True/False patches, such as erosion and dilation |
| erosion / dilation (of a mask) | Removing / adding one ring of cells around every patch |
| opening | Erosion followed by dilation: removes specks and thin parts |
| label / connected group | Numbering each set of touching True cells / one such set |
| 8-connectivity | Cells count as touching through edges and corners |
| plausibility rule | A stated rule that removes results that cannot be what you look for, such as change high above the valley floor |
| height above the valley floor | Here: elevation minus the lowest elevation within about 500 m |
| centreline | A line along the middle of a long shape, used to measure its length |
| deposition / erosion (by a river) | Sediment laid down / ground scoured away |
| field validation | Checking a map against observations made on the ground |

## Next

**D4 · Land-cover classification:** let the computer sort every cell of the valley into classes such as forest, fields, bare ground and water, and learn how to judge whether such a map can be trusted.

<!-- pygile-nav -->
**PyGILE Course** · lesson 36 of 39 · [Course home](../README.md)  
← Previous: [D2 · Spectral indices: vegetation, water and bare ground in the Melamchi valley](D02_Spectral_indices.ipynb) · Next: [D4 · Land-cover classification: forest, fields, sediment and water](D04_Land_cover_classification.ipynb) →